# LTL-Net — 基线对比 (Unet + Linknet)

跑两个卷积基线, 摸清最强 backbone。

**运行前**：右侧 Add Input 添加数据集 `yuanssy/v5data`（内含 datasetv5_random811）。

**注意**：首次训练会自动下载 resnet50 ImageNet 权重（约 100MB），两个模型共用同一个 backbone。

In [ ]:
import torch
print(f'PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

In [ ]:
!pip install rasterio segmentation-models-pytorch -q

In [ ]:
!rm -rf /kaggle/working/lunar-linear
!git clone https://github.com/song110585-cpu/lunar-linear.git /kaggle/working/lunar-linear
!cd /kaggle/working/lunar-linear && git checkout test-new-module

In [ ]:
import os
DATA_ROOT = '/kaggle/input/datasets/yuanssy/v5data/datasetv5_random811'
print('数据集存在:', os.path.isdir(DATA_ROOT))
if os.path.isdir(DATA_ROOT):
    for s in ['train', 'val', 'test']:
        img_dir = os.path.join(DATA_ROOT, s, 'image')
        n = len(os.listdir(img_dir)) if os.path.isdir(img_dir) else 0
        print(f'  {s}: {n} tiles')

In [ ]:
# ==== 跑 Unet 基线 ====
!cd /kaggle/working/lunar-linear/LTL-Net && python scripts/train_baseline.py --model Unet

In [ ]:
# ==== 跑 Linknet 基线 ====
!cd /kaggle/working/lunar-linear/LTL-Net && python scripts/train_baseline.py --model Linknet

In [ ]:
# ==== 打包结果 ====
import zipfile, os, glob
for d in sorted(glob.glob('/kaggle/working/result_*')):
    zip_path = d + '.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(d):
            for f in files:
                fpath = os.path.join(root, f)
                zf.write(fpath, os.path.relpath(fpath, '/kaggle/working'))
    print(f'{d} -> {zip_path}  ({os.path.getsize(zip_path)/1024:.0f} KB)')